# 🔵 Module 6: K-Nearest Neighbors (KNN) & Naive Bayes
## Theory + Practical — Complete Guide

---

## 📚 THEORY SECTION — K-Nearest Neighbors

### What is KNN?

**KNN** is a simple, non-parametric algorithm that classifies a new point based on the majority class of its K nearest neighbors in the training data.

```
Algorithm:
1. Choose K (number of neighbors)
2. For each new point:
   a. Compute distance to ALL training points
   b. Find K nearest points
   c. Predict: majority class (classification) or mean (regression)
```

### Distance Metrics

| Metric | Formula | When |
|--------|---------|------|
| **Euclidean** | √Σ(xᵢ-yᵢ)² | Continuous features |
| **Manhattan** | Σ|xᵢ-yᵢ| | Robust to outliers |
| **Minkowski** | (Σ|xᵢ-yᵢ|ᵖ)^(1/p) | Generalization (p=2→Euclidean, p=1→Manhattan) |
| **Cosine** | 1 - (x·y)/(‖x‖‖y‖) | Text, high-dimensional data |

### Choosing K

```
K too small → Overfitting (decision boundary very jagged)
K too large → Underfitting (decision boundary too smooth)

Rule of thumb: K = √n (where n = training set size)
```

### Pros & Cons of KNN

| Pros | Cons |
|------|------|
| Simple, intuitive | Slow at prediction (O(n) per query) |
| No training phase | High memory usage |
| Naturally multi-class | Sensitive to irrelevant features |
| Non-parametric | Requires feature scaling |

---

## 📚 THEORY SECTION — Naive Bayes

### Bayes' Theorem

```
P(class | features) = P(features | class) × P(class) / P(features)

Posterior = Likelihood × Prior / Evidence
```

### The 'Naive' Assumption

**Assumption**: All features are **conditionally independent** given the class.

```
P(x₁,x₂,...,xₙ | class) = P(x₁|class) × P(x₂|class) × ... × P(xₙ|class)
```

### Types of Naive Bayes

| Type | Feature Distribution | Use Case |
|------|---------------------|----------|
| **Gaussian NB** | Normal distribution | Continuous features |
| **Multinomial NB** | Count distribution | Text classification, word counts |
| **Bernoulli NB** | Binary distribution | Binary features |

### Why Naive Bayes Works Despite Wrong Assumption?

Even though feature independence is almost never true in practice, Naive Bayes often works well because:
- It's robust with small training data
- It works remarkably well for text classification
- The classification decision only needs the ranking of probabilities, not their exact values

---

## 🔬 PRACTICAL SECTION

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.pipeline import make_pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print('✅ Libraries loaded!')

### 🔬 Practical 1: KNN — Visualizing K Effect

In [ ]:
# ============================================================
# PRACTICAL 1: KNN — How K affects decision boundary
# ============================================================

iris = load_iris()
# Use only 2 features for visualization
X = iris.data[:, 2:4]  # Petal length, Petal width
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

k_values = [1, 3, 7, 15, 31]
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle('KNN: Effect of K on Decision Boundary', fontsize=14, fontweight='bold')

h = 0.05
x_min, x_max = X_train_s[:, 0].min() - 1, X_train_s[:, 0].max() + 1
y_min, y_max = X_train_s[:, 1].min() - 1, X_train_s[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

colors_map = {0: '#FF6B6B', 1: '#4ECDC4', 2: '#45B7D1'}
species_names = ['Setosa', 'Versicolor', 'Virginica']

for ax, k in zip(axes, k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    test_acc = accuracy_score(y_test, knn.predict(X_test_s))
    
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlBu')
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5)
    
    for cls in range(3):
        mask = y_train == cls
        ax.scatter(X_train_s[mask, 0], X_train_s[mask, 1], 
                   c=list(colors_map.values())[cls], s=40, edgecolors='k', 
                   linewidth=0.5, label=species_names[cls], alpha=0.8)
    
    ax.set_title(f'K={k}\nTest Acc={test_acc:.3f}', fontweight='bold')
    if k == 1:
        ax.text(0.5, -0.15, '← OVERFITTING', transform=ax.transAxes, 
                ha='center', color='red', fontsize=9)
    if k == 31:
        ax.text(0.5, -0.15, 'UNDERFITTING →', transform=ax.transAxes, 
                ha='center', color='orange', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Finding optimal K with cross-validation
# ============================================================

k_range = range(1, 51)
train_scores, cv_scores = [], []

for k in k_range:
    knn_pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    cv_score = cross_val_score(knn_pipe, X, y, cv=5, scoring='accuracy').mean()
    cv_scores.append(cv_score)

optimal_k = k_range[np.argmax(cv_scores)]

plt.figure(figsize=(12, 5))
plt.plot(k_range, cv_scores, 'bo-', linewidth=2, markersize=5, label='CV Score')
plt.axvline(optimal_k, color='red', linestyle='--', linewidth=2,
             label=f'Optimal K={optimal_k} (CV={max(cv_scores):.4f})')
plt.fill_between(k_range, cv_scores, alpha=0.1, color='blue')
plt.title('KNN: Finding Optimal K via Cross-Validation', fontweight='bold', fontsize=13)
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Cross-Validation Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print(f'✅ Optimal K = {optimal_k}, CV Score = {max(cv_scores):.4f}')

### 🔬 Practical 2: Naive Bayes — Text Spam Detection

In [ ]:
# ============================================================
# PRACTICAL 2: Naive Bayes — Spam Email Detection
# ============================================================

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Synthetic email dataset
spam_emails = [
    'Win free money now click here', 'Congratulations you won prize',
    'Free offer limited time act now', 'Get rich quick guaranteed money',
    'Nigerian prince transfer million dollars', 'Buy cheap pills online discount',
    'Earn cash from home easy money', 'Hot singles in your area free',
    'You have been selected winner claim prize', 'Double your income guaranteed',
    'Free casino bonus click now win', 'Make money fast from home today',
    'Special offer only for you free gift', 'Increase your revenue fast',
    'Free trial limited offer today only'
]

ham_emails = [
    'Meeting scheduled for tomorrow morning', 'Project update attached please review',
    'Quarterly report is ready for review', 'Team lunch at noon on Friday',
    'Please find the invoice attached', 'Conference call at 3pm today',
    'Code review comments addressed and merged', 'Happy birthday hope you have a great day',
    'Updated the documentation as requested', 'Your package has been shipped',
    'Resume for senior data scientist position', 'Let us discuss the project timeline',
    'Great work on the presentation today', 'Budget report for Q3 is attached',
    'Interview scheduled for next Tuesday'
]

emails = spam_emails + ham_emails
labels = [1] * len(spam_emails) + [0] * len(ham_emails)  # 1=spam, 0=ham

# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=50)
X_email = vectorizer.fit_transform(emails).toarray()
y_email = np.array(labels)

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_email, y_email, test_size=0.3, random_state=42
)

# Train Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train_e, y_train_e)
y_pred_e = gnb.predict(X_test_e)

print('=== Spam Email Detection with Gaussian Naive Bayes ===')
print(f'Test Accuracy: {accuracy_score(y_test_e, y_pred_e):.4f}')
print()
print(classification_report(y_test_e, y_pred_e, target_names=['Ham', 'Spam']))

# Test on new emails
test_messages = [
    'Win free money click now',
    'Schedule meeting for next week',
    'Congratulations you won a prize',
    'Please review the attached report'
]

print('\n=== Predictions on New Messages ===')
for msg in test_messages:
    X_msg = vectorizer.transform([msg]).toarray()
    pred = gnb.predict(X_msg)[0]
    prob = gnb.predict_proba(X_msg)[0]
    label = '🚨 SPAM' if pred == 1 else '✅ HAM'
    print(f'{label} ({prob[1]*100:.1f}% spam prob) | "{msg}"')

### 🔬 Practical 3: Algorithm Comparison

In [ ]:
# ============================================================
# PRACTICAL 3: Compare KNN and Naive Bayes on multiple datasets
# ============================================================

from sklearn.datasets import make_moons, make_circles, make_classification

datasets = [
    (make_moons(n_samples=300, noise=0.1, random_state=42), 'Moons'),
    (make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42), 'Circles'),
    (make_classification(n_samples=300, n_features=2, n_redundant=0, 
                         n_clusters_per_class=1, random_state=42), 'Linear'),
]

classifiers = [
    ('KNN (k=5)', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))),
    ('KNN (k=15)', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
    ('Gaussian NB', GaussianNB()),
]

fig, axes = plt.subplots(len(datasets), len(classifiers), figsize=(18, 12))
fig.suptitle('KNN vs Naive Bayes on Different Datasets', fontsize=14, fontweight='bold')

for row, ((X_d, y_d), dname) in enumerate(datasets):
    X_tr, X_te, y_tr, y_te = train_test_split(X_d, y_d, test_size=0.3, random_state=42)
    
    for col, (cname, clf) in enumerate(classifiers):
        ax = axes[row, col]
        clf.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, clf.predict(X_te))
        
        # Plot
        h = 0.05
        x_min, x_max = X_d[:, 0].min() - 0.5, X_d[:, 0].max() + 0.5
        y_min, y_max = X_d[:, 1].min() - 0.5, X_d[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
        Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        
        ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
        ax.scatter(X_d[:, 0], X_d[:, 1], c=y_d, cmap='RdYlBu', 
                   edgecolors='k', s=20, linewidth=0.3)
        ax.set_title(f'{dname}\n{cname}\nAcc={acc:.3f}', fontsize=9, fontweight='bold')
        ax.axis('off')

plt.tight_layout()
plt.show()

## 🎓 Summary

| Algorithm | Key Concept | Best For |
|-----------|------------|----------|
| KNN | Distance to nearest neighbors | Small datasets, non-linear boundaries |
| K=1 | Overfitting risk | - |
| Large K | Underfitting risk | - |
| Naive Bayes | Bayes' theorem + independence assumption | Text classification, high-dim sparse data |
| Gaussian NB | Continuous features | General classification |
| Multinomial NB | Count data | Text/document classification |
| Scaling | Critical for KNN | Always scale before KNN |

## 📖 Next: K-Means & Unsupervised Learning